# FinHallu-10K — run on Kaggle (2x T4, internet ON)
Runs 3 open models on 182 questions over 10 SEC 10-K filings. Results land in /kaggle/working/finhallu/results/.

In [ ]:
!pip install -q rank_bm25 beautifulsoup4 lxml openai pyyaml
import os
os.makedirs('/kaggle/working/finhallu/finhallu', exist_ok=True)
os.makedirs('/kaggle/working/finhallu/data', exist_ok=True)
%cd /kaggle/working/finhallu

In [ ]:
%%writefile finhallu/__init__.py
"""FinHallu-10K: LLM hallucination benchmark on SEC 10-K filings."""


In [ ]:
%%writefile finhallu/questions.py
"""Build the benchmark question set deterministically from SEC XBRL gold facts.

Five question types:
  lookup      - a single reported figure for the latest fiscal year
  prior_year  - the same figure for the prior year (tests year confusion)
  yoy         - year-over-year % change (tests arithmetic on retrieved numbers)
  ratio       - margin computed from two figures (tests multi-fact reasoning)
  trap        - unanswerable from the filing (tests fabrication vs. abstention)
  multiturn   - a follow-up that depends on the previous turn (tests context carry)
"""
from __future__ import annotations

import json
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent

LABELS = {
    "revenue": "total revenue (net sales)",
    "net_income": "net income",
    "total_assets": "total assets",
    "equity": "total shareholders' equity",
    "op_cash_flow": "net cash provided by (used in) operating activities",
    "rnd": "research and development expense",
    "eps_diluted": "diluted earnings per share",
    "cash": "cash and cash equivalents",
    "operating_income": "operating income",
}
BALANCE_SHEET = {"total_assets", "equity", "cash"}  # point-in-time items


def _when(m: str, label: str, fy_end: str, prior: bool = False) -> str:
    if m in BALANCE_SHEET:
        return f"as of the end of {label}"
    return f"for {label}"


def build(gold_path: Path = ROOT / "data" / "gold_facts.json") -> list[dict]:
    gold = json.loads(gold_path.read_text())["companies"]
    qs: list[dict] = []

    def add(**q):
        q["id"] = f"{q['ticker']}-{q['type']}-{sum(1 for x in qs if x['ticker'] == q['ticker'] and x['type'] == q['type']) + 1:02d}"
        qs.append(q)

    for tk, c in gold.items():
        name, fy, prior, mets = c["name"], c["fy_label"], c["prior_label"], c["metrics"]
        unit_of = lambda m: "usd_per_share" if m == "eps_diluted" else "usd_millions"

        # 1. lookups (latest year), all available metrics
        for m, (cur, _) in mets.items():
            add(ticker=tk, type="lookup", metric=m, unit=unit_of(m), answer=cur, prior_answer=mets[m][1],
                turns=[f"What was {name}'s {LABELS[m]} {_when(m, fy, c['fy_end'])}?"])

        # 2. prior-year lookups for 3 metrics
        for m in [x for x in ("revenue", "net_income", "op_cash_flow", "total_assets") if x in mets][:3]:
            add(ticker=tk, type="prior_year", metric=m, unit=unit_of(m), answer=mets[m][1], prior_answer=mets[m][0],
                turns=[f"What was {name}'s {LABELS[m]} {_when(m, prior, c['fy_end'])}, as reported in the comparative figures?"])

        # 3. YoY % change for up to 3 flow metrics
        for m in [x for x in ("revenue", "net_income", "operating_income", "op_cash_flow", "rnd") if x in mets][:3]:
            cur, prv = mets[m]
            add(ticker=tk, type="yoy", metric=m, unit="percent", answer=round((cur - prv) / abs(prv) * 100, 2),
                turns=[f"By what percentage did {name}'s {LABELS[m]} change from {prior} to {fy}? Give the percentage change."])

        # 4. margins
        for num in ("net_income", "operating_income"):
            if num in mets and "revenue" in mets:
                add(ticker=tk, type="ratio", metric=f"{num}_margin", unit="percent",
                    answer=round(mets[num][0] / mets["revenue"][0] * 100, 2),
                    turns=[f"What was {name}'s {LABELS[num]} as a percentage of total revenue {_when(num, fy, c['fy_end'])}?"])

        # 5. traps: information that cannot be in this filing
        fy_num = int(fy[-4:]) + 3  # three years after the filing's fiscal year
        add(ticker=tk, type="trap", metric="future_revenue", unit="abstain", answer=None,
            turns=[f"What was {name}'s actual reported total revenue for fiscal year {fy_num}?"])
        add(ticker=tk, type="trap", metric="future_quarter", unit="abstain", answer=None,
            turns=[f"What diluted EPS did {name} report for the third quarter of fiscal year {fy_num}?"])

        # 6. multi-turn follow-up (graded on the last turn)
        if "revenue" in mets:
            add(ticker=tk, type="multiturn", metric="revenue", unit="usd_millions", answer=mets["revenue"][1],
                prior_answer=mets["revenue"][0],
                turns=[f"What was {name}'s {LABELS['revenue']} {_when('revenue', fy, c['fy_end'])}?",
                       "And what was it in the prior fiscal year?"])
    return qs


def main() -> None:
    qs = build()
    out = ROOT / "data" / "questions.jsonl"
    out.write_text("\n".join(json.dumps(q) for q in qs) + "\n")
    from collections import Counter
    print(f"wrote {len(qs)} questions -> {out}")
    print(dict(Counter(q["type"] for q in qs)))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile finhallu/corpus.py
"""Download the 10 filings from SEC EDGAR, convert to plain text, chunk, and index with BM25.

SEC asks automated clients to send a descriptive User-Agent with contact info and stay under
10 requests/second. Set SEC_USER_AGENT="Your Name you@example.com" before running.
"""
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
RAW = ROOT / "data" / "filings_raw"
TXT = ROOT / "data" / "filings_txt"


def filing_url(c: dict) -> str:
    return f"https://www.sec.gov/Archives/edgar/data/{c['cik']}/{c['accn'].replace('-', '')}/{c['doc']}"


def html_to_text(html: str) -> str:
    """Inline-XBRL 10-K HTML -> readable text, keeping table rows on one line."""
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style", "head"]):
        t.decompose()
    for hidden in soup.find_all(style=re.compile(r"display:\s*none", re.I)):
        hidden.decompose()  # the ix:header block of hidden XBRL facts
    for tr in soup.find_all("tr"):
        cells = [c.get_text(" ", strip=True) for c in tr.find_all(["td", "th"])]
        cells = [c for c in cells if c and c not in {"$", ")", "%"}]
        tr.replace_with(soup.new_string("\n" + " | ".join(cells) + "\n"))
    text = soup.get_text("\n")
    text = re.sub(r"[ \t\xa0]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)
    return text.strip()


def download(gold_path: Path = ROOT / "data" / "gold_facts.json") -> None:
    import requests

    ua = os.environ.get("SEC_USER_AGENT")
    if not ua:
        raise SystemExit('Set SEC_USER_AGENT="Your Name your@email.com" (SEC requires a contact User-Agent).')
    RAW.mkdir(parents=True, exist_ok=True)
    TXT.mkdir(parents=True, exist_ok=True)
    for tk, c in json.loads(gold_path.read_text())["companies"].items():
        raw = RAW / f"{tk}.htm"
        if not raw.exists():
            r = requests.get(filing_url(c), headers={"User-Agent": ua}, timeout=60)
            r.raise_for_status()
            raw.write_bytes(r.content)
            time.sleep(0.5)
        txt = TXT / f"{tk}.txt"
        txt.write_text(html_to_text(raw.read_text(errors="ignore")))
        print(f"{tk}: {txt.stat().st_size // 1024} KB text")


def chunk(text: str, size: int = 1500, overlap: int = 250) -> list[str]:
    out, i = [], 0
    while i < len(text):
        out.append(text[i : i + size])
        i += size - overlap
    return out


_tok = re.compile(r"[a-z0-9]+")


def tokenize(s: str) -> list[str]:
    return _tok.findall(s.lower())


class Retriever:
    """BM25 over one filing's chunks — the cheap baseline most startups actually ship."""

    def __init__(self, text: str, k: int = 8):
        from rank_bm25 import BM25Okapi

        self.chunks = chunk(text)
        self.bm25 = BM25Okapi([tokenize(c) for c in self.chunks])
        self.k = k

    def search(self, query: str) -> list[str]:
        scores = self.bm25.get_scores(tokenize(query))
        top = sorted(range(len(scores)), key=lambda i: -scores[i])[: self.k]
        return [self.chunks[i] for i in sorted(top)]  # keep document order


def load_retrievers(k: int = 8) -> dict[str, Retriever]:
    if not TXT.exists() or not any(TXT.glob("*.txt")):
        raise SystemExit("No filings yet. Run: python -m finhallu.corpus")
    return {p.stem: Retriever(p.read_text(), k=k) for p in sorted(TXT.glob("*.txt"))}


if __name__ == "__main__":
    download()


In [ ]:
%%writefile finhallu/grade.py
"""Deterministic grading + failure taxonomy.

Every answer gets one label:
  correct           - within tolerance of the gold value (or abstained on a trap)
  fabricated        - gave a number for an unanswerable (trap) question
  wrong_year        - matches the other fiscal year's value
  scale_error       - right digits, wrong unit (x1000 / /1000, e.g. thousands vs millions)
  sign_error        - right magnitude, wrong sign (e.g. cash used vs provided)
  wrong_value       - a number that matches nothing in the gold set for that question
  false_refusal     - abstained although the answer is in the filing
  no_answer         - no parsable number and no clear abstention
"""
from __future__ import annotations

import re

ABSTAIN = "NOT_IN_DOCUMENT"
_abstain_re = re.compile(
    r"not_in_document|not (?:provided|available|disclosed|included|mentioned|found|stated|contain|reported)|"
    r"(?:does|do|did) not (?:contain|include|provide|report|disclose|mention)|cannot (?:be )?(?:determine|find|answer)|"
    r"no (?:information|data|mention)|unable to (?:determine|find|answer)|isn't (?:in|provided|available)",
    re.I,
)
_num_re = re.compile(
    r"(?P<neg>[-−(])?\s*\$?\s*(?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)\s*(?P<scale>trillion|billion|million|thousand|bn|mn|[tbmk](?![a-z]))?\s*(?P<pct>%|percent)?",
    re.I,
)
_SCALE = {"trillion": 1e6, "t": 1e6, "billion": 1e3, "bn": 1e3, "b": 1e3, "million": 1, "mn": 1, "m": 1, "thousand": 1e-3, "k": 1e-3}


def abstained(text: str) -> bool:
    return bool(_abstain_re.search(text or ""))


def extract_number(text: str, unit: str) -> float | None:
    """Pull the answer number from the FINAL ANSWER line if present, else the first number.

    For usd_millions answers, the number is converted to millions using any scale word.
    A bare number with no scale word is assumed to already be in millions (the filings' unit).
    """
    if not text:
        return None
    m = re.search(r"final answer\s*[:\-]\s*(.+)", text, re.I)
    seg = m.group(1) if m else text
    for mt in _num_re.finditer(seg):
        raw = mt.group("num").replace(",", "")
        if re.fullmatch(r"(19|20)\d\d", raw) and not mt.group("scale") and not mt.group("pct"):
            continue  # skip years
        val = float(raw)
        if mt.group("neg"):
            val = -val
        if unit == "percent" and val > 0 and re.search(r"decrease|decline|fell|drop|down|lower|reduc", seg[: mt.start()] + seg[mt.end(): mt.end() + 20], re.I):
            val = -val
        sc = (mt.group("scale") or "").lower()
        if unit == "usd_millions":
            if sc in _SCALE:
                val *= _SCALE[sc]
            elif val > 1e8:  # raw dollars written out in full
                val /= 1e6
        return val
    return None


def _close(a: float, b: float, unit: str) -> bool:
    if unit == "percent":
        return abs(a - b) <= 0.5  # percentage points
    if unit == "usd_per_share":
        return abs(a - b) <= 0.011
    return abs(a - b) <= max(abs(b) * 0.005, 0.6)  # 0.5% handles "$416.2 billion" style rounding


def grade(q: dict, answer: str) -> str:
    unit, gold = q["unit"], q["answer"]
    if unit == "abstain":
        if extract_number_for_trap(answer) and not abstained(answer):
            return "fabricated"
        return "correct" if abstained(answer) else "no_answer"
    val = extract_number(answer, unit)
    if val is None:
        return "false_refusal" if abstained(answer) else "no_answer"
    if _close(val, gold, unit):
        return "correct"
    other = q.get("prior_answer")
    if other is not None and _close(val, other, unit):
        return "wrong_year"
    if unit == "usd_millions" and any(_close(val * f, gold, unit) for f in (1e3, 1e-3, 1e6, 1e-6)):
        return "scale_error"
    if _close(-val, gold, unit):
        return "sign_error"
    return "wrong_value"


def extract_number_for_trap(text: str) -> bool:
    """True if the answer asserts a monetary/EPS figure (ignoring years)."""
    return extract_number(text or "", "usd_millions") is not None


In [ ]:
%%writefile finhallu/models.py
"""Model clients. Anything OpenAI-compatible works: OpenAI, OpenRouter, Together, Groq, or a local vLLM server.

Configure in models.yaml. API keys are read from the env var named in `api_key_env`; never commit them.
"""
from __future__ import annotations

import os
import time
from dataclasses import dataclass, field

SYSTEM = (
    "You are a financial analyst assistant. Answer ONLY from the 10-K excerpts provided. "
    "Monetary amounts in these filings are usually in millions of US dollars. "
    "If the excerpts do not contain the answer, reply exactly: NOT_IN_DOCUMENT. "
    "End with one line: 'FINAL ANSWER: <number with units, or NOT_IN_DOCUMENT>'."
)


@dataclass
class ModelSpec:
    name: str
    model: str
    base_url: str | None = None
    api_key_env: str = "OPENAI_API_KEY"
    temperature: float = 0.0
    max_tokens: int = 400
    extra: dict = field(default_factory=dict)


class ChatModel:
    def __init__(self, spec: ModelSpec):
        from openai import OpenAI

        self.spec = spec
        key = os.environ.get(spec.api_key_env, "EMPTY")  # local vLLM accepts any key
        self.client = OpenAI(base_url=spec.base_url, api_key=key)

    def chat(self, messages: list[dict], qid: str | None = None) -> str:
        for attempt in range(4):
            try:
                r = self.client.chat.completions.create(
                    model=self.spec.model, messages=messages,
                    temperature=self.spec.temperature, max_tokens=self.spec.max_tokens, **self.spec.extra,
                )
                return r.choices[0].message.content or ""
            except Exception as e:  # rate limits, transient 5xx
                if attempt == 3:
                    return f"[ERROR] {type(e).__name__}: {e}"
                time.sleep(2 ** attempt * 2)
        return ""


class OracleModel:
    """Test double: answers from gold. Used by tests and `--dry-run` to validate the pipeline."""

    def __init__(self, questions: dict[str, dict], noise: bool = False):
        self.q, self.noise, self.spec = questions, noise, ModelSpec(name="oracle" + ("-noisy" if noise else ""), model="oracle")

    def chat(self, messages: list[dict], qid: str | None = None) -> str:
        q = self.q[qid]
        if q["unit"] == "abstain":
            return "FINAL ANSWER: $512 million" if self.noise else "FINAL ANSWER: NOT_IN_DOCUMENT"
        v = q.get("prior_answer") if (self.noise and q.get("prior_answer") is not None) else q["answer"]
        suffix = {"usd_millions": " million", "percent": "%", "usd_per_share": ""}[q["unit"]]
        return f"FINAL ANSWER: ${v:,}{suffix}" if q["unit"] != "percent" else f"FINAL ANSWER: {v}%"


In [ ]:
%%writefile finhallu/hf_local.py
"""Run open models locally with Hugging Face transformers (no API key, no server).

Works on a free Kaggle/Colab T4 for models up to ~8B in float16 (use 2xT4 on Kaggle for 7-8B).
"""
from __future__ import annotations

from .models import ModelSpec


class HFModel:
    def __init__(self, model_id: str, max_new_tokens: int = 300):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.spec = ModelSpec(name=model_id.split("/")[-1], model=model_id)
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
        self.model.eval()
        self.max_new_tokens = max_new_tokens

    def chat(self, messages: list[dict], qid: str | None = None) -> str:
        import torch

        try:
            ids = self.tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
        except Exception:  # templates without a system role: fold it into the first user turn
            sys_msg, rest = messages[0]["content"], messages[1:]
            rest = [dict(rest[0], content=sys_msg + "\n\n" + rest[0]["content"])] + rest[1:]
            ids = self.tok.apply_chat_template(rest, add_generation_prompt=True, return_tensors="pt")
        ids = ids.to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(ids, max_new_tokens=self.max_new_tokens, do_sample=False,
                                      pad_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

    def close(self) -> None:
        import gc

        import torch

        del self.model
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
%%writefile finhallu/run.py
"""Run every model on every question with BM25 retrieval, grade, and save results.

    python -m finhallu.run                  # all models in models.yaml
    python -m finhallu.run --models gpt-4.1-mini qwen-local
    python -m finhallu.run --dry-run        # oracle models, no API calls, no filings needed
"""
from __future__ import annotations

import argparse
import csv
import json
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from .grade import grade
from .models import SYSTEM, ChatModel, ModelSpec, OracleModel

ROOT = Path(__file__).resolve().parent.parent
RESULTS = ROOT / "results"


def load_questions() -> list[dict]:
    return [json.loads(l) for l in (ROOT / "data" / "questions.jsonl").read_text().splitlines() if l.strip()]


def load_specs(names: list[str] | None) -> list[ModelSpec]:
    import yaml

    cfg = yaml.safe_load((ROOT / "models.yaml").read_text())
    specs = [ModelSpec(**m) for m in cfg["models"]]
    return [s for s in specs if not names or s.name in names]


def ask(model, q: dict, retr) -> dict:
    """Retrieve per turn, carry the conversation, grade the last turn."""
    msgs = [{"role": "system", "content": SYSTEM}]
    answer, ctx_hit = "", None
    for i, turn in enumerate(q["turns"]):
        query = " ".join(q["turns"][: i + 1])  # follow-ups retrieve with the earlier turn as context
        ctx = "\n\n---\n\n".join(retr.search(query)) if retr else ""
        msgs.append({"role": "user", "content": f"10-K excerpts:\n{ctx}\n\nQuestion: {turn}"})
        answer = model.chat(msgs, qid=q["id"])
        msgs.append({"role": "assistant", "content": answer})
    return {
        "id": q["id"], "ticker": q["ticker"], "type": q["type"], "metric": q["metric"], "unit": q["unit"],
        "gold": q["answer"], "model": model.spec.name, "answer": answer.strip(), "label": grade(q, answer),
    }


def run(models, questions, retrievers, workers: int = 6) -> list[dict]:
    rows = []
    for m in models:
        with ThreadPoolExecutor(workers) as ex:
            futs = [ex.submit(ask, m, q, retrievers.get(q["ticker"]) if retrievers else None) for q in questions]
            for f in futs:
                rows.append(f.result())
        acc = sum(r["label"] == "correct" for r in rows if r["model"] == m.spec.name) / len(questions)
        print(f"{m.spec.name:28s} accuracy {acc:6.1%}")
    return rows


def save(rows: list[dict], tag: str = "") -> Path:
    RESULTS.mkdir(exist_ok=True)
    out = RESULTS / f"results{tag}.csv"
    with out.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"saved {len(rows)} rows -> {out}")
    return out


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", nargs="*")
    ap.add_argument("--dry-run", action="store_true")
    ap.add_argument("--k", type=int, default=8, help="chunks retrieved per question")
    ap.add_argument("--limit", type=int, help="first N questions only (smoke test)")
    ap.add_argument("--hf", nargs="*", help="Hugging Face model ids to run locally on GPU instead of API models")
    a = ap.parse_args()

    qs = load_questions()[: a.limit] if a.limit else load_questions()
    if a.dry_run:
        by_id = {q["id"]: q for q in qs}
        rows = run([OracleModel(by_id), OracleModel(by_id, noise=True)], qs, None)
        save(rows, "_dryrun")
        return
    from .corpus import load_retrievers

    retr = load_retrievers(a.k)
    if a.hf:  # local GPU models, loaded one at a time to fit in memory
        from .hf_local import HFModel

        rows = []
        for mid in a.hf:
            m = HFModel(mid)
            rows += run([m], qs, retr, workers=1)
            save(rows)  # checkpoint after each model
            m.close()
        return
    rows = run([ChatModel(s) for s in load_specs(a.models)], qs, retr)
    save(rows)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile finhallu/report.py
"""Turn results CSV into: summary table (markdown), failure taxonomy, and a chart for the post.

    python -m finhallu.report                       # results/results.csv
    python -m finhallu.report results/results_dryrun.csv
"""
from __future__ import annotations

import csv
import sys
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
TYPES = ["lookup", "prior_year", "yoy", "ratio", "multiturn", "trap"]
LABELS = ["correct", "fabricated", "wrong_year", "scale_error", "sign_error", "wrong_value", "false_refusal", "no_answer"]


def main(path: str | None = None) -> None:
    p = Path(path) if path else ROOT / "results" / "results.csv"
    rows = list(csv.DictReader(p.open()))
    models = sorted({r["model"] for r in rows})
    acc = defaultdict(lambda: defaultdict(list))
    fails = defaultdict(Counter)
    for r in rows:
        acc[r["model"]][r["type"]].append(r["label"] == "correct")
        acc[r["model"]]["overall"].append(r["label"] == "correct")
        if r["label"] != "correct":
            fails[r["model"]][r["label"]] += 1

    pct = lambda xs: f"{100 * sum(xs) / len(xs):.0f}%" if xs else "–"
    md = ["## Accuracy by question type", "", "| Model | Overall | " + " | ".join(TYPES) + " |", "|---" * (len(TYPES) + 2) + "|"]
    for m in sorted(models, key=lambda m: -sum(acc[m]["overall"])):
        md.append(f"| {m} | **{pct(acc[m]['overall'])}** | " + " | ".join(pct(acc[m][t]) for t in TYPES) + " |")
    md += ["", "## Failure taxonomy (count of wrong answers)", "", "| Model | " + " | ".join(LABELS[1:]) + " |", "|---" * len(LABELS) + "|"]
    for m in models:
        md.append(f"| {m} | " + " | ".join(str(fails[m][l]) for l in LABELS[1:]) + " |")

    # trap fabrication rate — usually the headline finding
    md += ["", "## Headline numbers", ""]
    for m in models:
        traps = acc[m]["trap"]
        if traps:
            md.append(f"- **{m}** invented a number on {100 - 100 * sum(traps) / len(traps):.0f}% of unanswerable questions.")
    out = p.with_name(p.stem + "_summary.md")
    out.write_text("\n".join(md) + "\n")
    print("\n".join(md))
    _chart(models, acc, p.with_name(p.stem + "_chart.png"))


def _chart(models, acc, out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4.8), dpi=160)
    w = 0.8 / max(len(models), 1)
    palette = ["#2a6fdb", "#e4572e", "#17a398", "#f3a712", "#7d5ba6", "#555555"]
    for i, m in enumerate(models):
        vals = [100 * sum(acc[m][t]) / len(acc[m][t]) if acc[m][t] else 0 for t in TYPES]
        ax.bar([x + i * w for x in range(len(TYPES))], vals, w, label=m, color=palette[i % len(palette)])
    ax.set_xticks([x + w * (len(models) - 1) / 2 for x in range(len(TYPES))])
    ax.set_xticklabels(["lookup", "prior year", "YoY %", "margin", "follow-up", "unanswerable\n(abstained)"])
    ax.set_ylabel("% correct")
    ax.set_ylim(0, 100)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, ncol=min(len(models), 3), loc="upper center", bbox_to_anchor=(0.5, 1.15))
    ax.set_title("10-K QA with BM25 retrieval: where LLMs get financial facts wrong", pad=34, loc="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(out)
    print(f"chart -> {out}")


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else None)


In [ ]:
%%writefile data/gold_facts.json
{
  "_source": "SEC EDGAR XBRL companyfacts API (data.sec.gov), facts tagged to each company's latest 10-K accession. Pulled 2026-10-13. Monetary values in USD millions; EPS in USD per diluted share.",
  "companies": {
    "AAPL": {"name": "Apple Inc.", "cik": 320193, "accn": "0000320193-25-000079", "doc": "aapl-20250927.htm", "fy_end": "2025-09-27", "fy_label": "fiscal 2025", "prior_label": "fiscal 2024",
      "metrics": {"revenue": [416161, 391035], "net_income": [112010, 93736], "total_assets": [359241, 364980], "equity": [73733, 56950], "op_cash_flow": [111482, 118254], "rnd": [34550, 31370], "eps_diluted": [7.46, 6.08], "cash": [35934, 29943], "operating_income": [133050, 123216]}},
    "MSFT": {"name": "Microsoft Corporation", "cik": 789019, "accn": "0001193125-26-323660", "doc": "msft-20260630.htm", "fy_end": "2026-06-30", "fy_label": "fiscal 2026", "prior_label": "fiscal 2025",
      "metrics": {"revenue": [331839, 281724], "net_income": [133749, 101832], "total_assets": [758376, 619003], "equity": [442387, 343479], "op_cash_flow": [182935, 136162], "rnd": [35562, 32488], "eps_diluted": [17.95, 13.64], "cash": [20935, 30242], "operating_income": [155237, 128528]}},
    "NVDA": {"name": "NVIDIA Corporation", "cik": 1045810, "accn": "0001045810-26-000021", "doc": "nvda-20260125.htm", "fy_end": "2026-01-25", "fy_label": "fiscal 2026", "prior_label": "fiscal 2025",
      "metrics": {"revenue": [215938, 130497], "net_income": [120067, 72880], "total_assets": [206803, 111601], "equity": [157293, 79327], "op_cash_flow": [102718, 64089], "rnd": [18497, 12914], "eps_diluted": [4.90, 2.94], "cash": [10605, 8589], "operating_income": [130387, 81453]}},
    "JPM": {"name": "JPMorgan Chase & Co.", "cik": 19617, "accn": "0001628280-26-008131", "doc": "jpm-20251231.htm", "fy_end": "2025-12-31", "fy_label": "2025", "prior_label": "2024",
      "metrics": {"revenue": [182447, 177556], "net_income": [57048, 58471], "total_assets": [4424900, 4002814], "equity": [362438, 344758], "op_cash_flow": [-147782, -42012], "eps_diluted": [20.02, 19.75]}},
    "XOM": {"name": "Exxon Mobil Corporation", "cik": 34088, "accn": "0000034088-26-000045", "doc": "xom-20251231.htm", "fy_end": "2025-12-31", "fy_label": "2025", "prior_label": "2024",
      "metrics": {"revenue": [332238, 349585], "net_income": [28844, 33680], "total_assets": [448980, 453475], "equity": [259386, 263705], "op_cash_flow": [51970, 55022], "eps_diluted": [6.70, 7.84], "cash": [10681, 23029]}},
    "JNJ": {"name": "Johnson & Johnson", "cik": 200406, "accn": "0000200406-26-000016", "doc": "jnj-20251228.htm", "fy_end": "2025-12-28", "fy_label": "fiscal 2025", "prior_label": "fiscal 2024",
      "metrics": {"revenue": [94193, 88821], "net_income": [26804, 14066], "total_assets": [199210, 180104], "op_cash_flow": [24530, 24266], "eps_diluted": [11.03, 5.79], "cash": [19709, 24105]}},
    "WMT": {"name": "Walmart Inc.", "cik": 104169, "accn": "0000104169-26-000055", "doc": "wmt-20260131.htm", "fy_end": "2026-01-31", "fy_label": "fiscal 2026", "prior_label": "fiscal 2025",
      "metrics": {"revenue": [706413, 674538], "net_income": [21893, 19436], "total_assets": [284668, 260823], "equity": [99617, 91013], "op_cash_flow": [41565, 36443], "cash": [10727, 9037], "operating_income": [29825, 29348]}},
    "CAT": {"name": "Caterpillar Inc.", "cik": 18230, "accn": "0000018230-26-000008", "doc": "cat-20251231.htm", "fy_end": "2025-12-31", "fy_label": "2025", "prior_label": "2024",
      "metrics": {"revenue": [67589, 64809], "total_assets": [98585, 87764], "op_cash_flow": [11739, 12035], "rnd": [2148, 2107], "eps_diluted": [18.81, 22.05], "cash": [9980, 6889], "operating_income": [11151, 13072]}},
    "KO": {"name": "The Coca-Cola Company", "cik": 21344, "accn": "0001628280-26-010047", "doc": "ko-20251231.htm", "fy_end": "2025-12-31", "fy_label": "2025", "prior_label": "2024",
      "metrics": {"revenue": [47941, 47061], "net_income": [13107, 10631], "total_assets": [104816, 100549], "equity": [32169, 24856], "op_cash_flow": [7408, 6805], "cash": [10270, 10828], "operating_income": [13762, 9992]}},
    "NFLX": {"name": "Netflix, Inc.", "cik": 1065280, "accn": "0001065280-26-000034", "doc": "nflx-20251231.htm", "fy_end": "2025-12-31", "fy_label": "2025", "prior_label": "2024",
      "metrics": {"revenue": [45183.036, 39000.966], "net_income": [10981.201, 8711.631], "total_assets": [55596.993, 53630.374], "equity": [26615.488, 24743.567], "op_cash_flow": [10149.273, 7361.364], "rnd": [3391.39, 2925.295], "eps_diluted": [2.53, 1.98], "cash": [9033.681, 7804.733], "operating_income": [13326.603, 10417.614]}}
  }
}


In [ ]:
%%writefile data/questions.jsonl
{"ticker": "AAPL", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 416161, "prior_answer": 391035, "turns": ["What was Apple Inc.'s total revenue (net sales) for fiscal 2025?"], "id": "AAPL-lookup-01"}
{"ticker": "AAPL", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 112010, "prior_answer": 93736, "turns": ["What was Apple Inc.'s net income for fiscal 2025?"], "id": "AAPL-lookup-02"}
{"ticker": "AAPL", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 359241, "prior_answer": 364980, "turns": ["What was Apple Inc.'s total assets as of the end of fiscal 2025?"], "id": "AAPL-lookup-03"}
{"ticker": "AAPL", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 73733, "prior_answer": 56950, "turns": ["What was Apple Inc.'s total shareholders' equity as of the end of fiscal 2025?"], "id": "AAPL-lookup-04"}
{"ticker": "AAPL", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 111482, "prior_answer": 118254, "turns": ["What was Apple Inc.'s net cash provided by (used in) operating activities for fiscal 2025?"], "id": "AAPL-lookup-05"}
{"ticker": "AAPL", "type": "lookup", "metric": "rnd", "unit": "usd_millions", "answer": 34550, "prior_answer": 31370, "turns": ["What was Apple Inc.'s research and development expense for fiscal 2025?"], "id": "AAPL-lookup-06"}
{"ticker": "AAPL", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 7.46, "prior_answer": 6.08, "turns": ["What was Apple Inc.'s diluted earnings per share for fiscal 2025?"], "id": "AAPL-lookup-07"}
{"ticker": "AAPL", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 35934, "prior_answer": 29943, "turns": ["What was Apple Inc.'s cash and cash equivalents as of the end of fiscal 2025?"], "id": "AAPL-lookup-08"}
{"ticker": "AAPL", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 133050, "prior_answer": 123216, "turns": ["What was Apple Inc.'s operating income for fiscal 2025?"], "id": "AAPL-lookup-09"}
{"ticker": "AAPL", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 391035, "prior_answer": 416161, "turns": ["What was Apple Inc.'s total revenue (net sales) for fiscal 2024, as reported in the comparative figures?"], "id": "AAPL-prior_year-01"}
{"ticker": "AAPL", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 93736, "prior_answer": 112010, "turns": ["What was Apple Inc.'s net income for fiscal 2024, as reported in the comparative figures?"], "id": "AAPL-prior_year-02"}
{"ticker": "AAPL", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 118254, "prior_answer": 111482, "turns": ["What was Apple Inc.'s net cash provided by (used in) operating activities for fiscal 2024, as reported in the comparative figures?"], "id": "AAPL-prior_year-03"}
{"ticker": "AAPL", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 6.43, "turns": ["By what percentage did Apple Inc.'s total revenue (net sales) change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "AAPL-yoy-01"}
{"ticker": "AAPL", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 19.5, "turns": ["By what percentage did Apple Inc.'s net income change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "AAPL-yoy-02"}
{"ticker": "AAPL", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 7.98, "turns": ["By what percentage did Apple Inc.'s operating income change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "AAPL-yoy-03"}
{"ticker": "AAPL", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 26.92, "turns": ["What was Apple Inc.'s net income as a percentage of total revenue for fiscal 2025?"], "id": "AAPL-ratio-01"}
{"ticker": "AAPL", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 31.97, "turns": ["What was Apple Inc.'s operating income as a percentage of total revenue for fiscal 2025?"], "id": "AAPL-ratio-02"}
{"ticker": "AAPL", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Apple Inc.'s actual reported total revenue for fiscal year 2028?"], "id": "AAPL-trap-01"}
{"ticker": "AAPL", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Apple Inc. report for the third quarter of fiscal year 2028?"], "id": "AAPL-trap-02"}
{"ticker": "AAPL", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 391035, "prior_answer": 416161, "turns": ["What was Apple Inc.'s total revenue (net sales) for fiscal 2025?", "And what was it in the prior fiscal year?"], "id": "AAPL-multiturn-01"}
{"ticker": "MSFT", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 331839, "prior_answer": 281724, "turns": ["What was Microsoft Corporation's total revenue (net sales) for fiscal 2026?"], "id": "MSFT-lookup-01"}
{"ticker": "MSFT", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 133749, "prior_answer": 101832, "turns": ["What was Microsoft Corporation's net income for fiscal 2026?"], "id": "MSFT-lookup-02"}
{"ticker": "MSFT", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 758376, "prior_answer": 619003, "turns": ["What was Microsoft Corporation's total assets as of the end of fiscal 2026?"], "id": "MSFT-lookup-03"}
{"ticker": "MSFT", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 442387, "prior_answer": 343479, "turns": ["What was Microsoft Corporation's total shareholders' equity as of the end of fiscal 2026?"], "id": "MSFT-lookup-04"}
{"ticker": "MSFT", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 182935, "prior_answer": 136162, "turns": ["What was Microsoft Corporation's net cash provided by (used in) operating activities for fiscal 2026?"], "id": "MSFT-lookup-05"}
{"ticker": "MSFT", "type": "lookup", "metric": "rnd", "unit": "usd_millions", "answer": 35562, "prior_answer": 32488, "turns": ["What was Microsoft Corporation's research and development expense for fiscal 2026?"], "id": "MSFT-lookup-06"}
{"ticker": "MSFT", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 17.95, "prior_answer": 13.64, "turns": ["What was Microsoft Corporation's diluted earnings per share for fiscal 2026?"], "id": "MSFT-lookup-07"}
{"ticker": "MSFT", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 20935, "prior_answer": 30242, "turns": ["What was Microsoft Corporation's cash and cash equivalents as of the end of fiscal 2026?"], "id": "MSFT-lookup-08"}
{"ticker": "MSFT", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 155237, "prior_answer": 128528, "turns": ["What was Microsoft Corporation's operating income for fiscal 2026?"], "id": "MSFT-lookup-09"}
{"ticker": "MSFT", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 281724, "prior_answer": 331839, "turns": ["What was Microsoft Corporation's total revenue (net sales) for fiscal 2025, as reported in the comparative figures?"], "id": "MSFT-prior_year-01"}
{"ticker": "MSFT", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 101832, "prior_answer": 133749, "turns": ["What was Microsoft Corporation's net income for fiscal 2025, as reported in the comparative figures?"], "id": "MSFT-prior_year-02"}
{"ticker": "MSFT", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 136162, "prior_answer": 182935, "turns": ["What was Microsoft Corporation's net cash provided by (used in) operating activities for fiscal 2025, as reported in the comparative figures?"], "id": "MSFT-prior_year-03"}
{"ticker": "MSFT", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 17.79, "turns": ["By what percentage did Microsoft Corporation's total revenue (net sales) change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "MSFT-yoy-01"}
{"ticker": "MSFT", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 31.34, "turns": ["By what percentage did Microsoft Corporation's net income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "MSFT-yoy-02"}
{"ticker": "MSFT", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 20.78, "turns": ["By what percentage did Microsoft Corporation's operating income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "MSFT-yoy-03"}
{"ticker": "MSFT", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 40.31, "turns": ["What was Microsoft Corporation's net income as a percentage of total revenue for fiscal 2026?"], "id": "MSFT-ratio-01"}
{"ticker": "MSFT", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 46.78, "turns": ["What was Microsoft Corporation's operating income as a percentage of total revenue for fiscal 2026?"], "id": "MSFT-ratio-02"}
{"ticker": "MSFT", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Microsoft Corporation's actual reported total revenue for fiscal year 2029?"], "id": "MSFT-trap-01"}
{"ticker": "MSFT", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Microsoft Corporation report for the third quarter of fiscal year 2029?"], "id": "MSFT-trap-02"}
{"ticker": "MSFT", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 281724, "prior_answer": 331839, "turns": ["What was Microsoft Corporation's total revenue (net sales) for fiscal 2026?", "And what was it in the prior fiscal year?"], "id": "MSFT-multiturn-01"}
{"ticker": "NVDA", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 215938, "prior_answer": 130497, "turns": ["What was NVIDIA Corporation's total revenue (net sales) for fiscal 2026?"], "id": "NVDA-lookup-01"}
{"ticker": "NVDA", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 120067, "prior_answer": 72880, "turns": ["What was NVIDIA Corporation's net income for fiscal 2026?"], "id": "NVDA-lookup-02"}
{"ticker": "NVDA", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 206803, "prior_answer": 111601, "turns": ["What was NVIDIA Corporation's total assets as of the end of fiscal 2026?"], "id": "NVDA-lookup-03"}
{"ticker": "NVDA", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 157293, "prior_answer": 79327, "turns": ["What was NVIDIA Corporation's total shareholders' equity as of the end of fiscal 2026?"], "id": "NVDA-lookup-04"}
{"ticker": "NVDA", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 102718, "prior_answer": 64089, "turns": ["What was NVIDIA Corporation's net cash provided by (used in) operating activities for fiscal 2026?"], "id": "NVDA-lookup-05"}
{"ticker": "NVDA", "type": "lookup", "metric": "rnd", "unit": "usd_millions", "answer": 18497, "prior_answer": 12914, "turns": ["What was NVIDIA Corporation's research and development expense for fiscal 2026?"], "id": "NVDA-lookup-06"}
{"ticker": "NVDA", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 4.9, "prior_answer": 2.94, "turns": ["What was NVIDIA Corporation's diluted earnings per share for fiscal 2026?"], "id": "NVDA-lookup-07"}
{"ticker": "NVDA", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 10605, "prior_answer": 8589, "turns": ["What was NVIDIA Corporation's cash and cash equivalents as of the end of fiscal 2026?"], "id": "NVDA-lookup-08"}
{"ticker": "NVDA", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 130387, "prior_answer": 81453, "turns": ["What was NVIDIA Corporation's operating income for fiscal 2026?"], "id": "NVDA-lookup-09"}
{"ticker": "NVDA", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 130497, "prior_answer": 215938, "turns": ["What was NVIDIA Corporation's total revenue (net sales) for fiscal 2025, as reported in the comparative figures?"], "id": "NVDA-prior_year-01"}
{"ticker": "NVDA", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 72880, "prior_answer": 120067, "turns": ["What was NVIDIA Corporation's net income for fiscal 2025, as reported in the comparative figures?"], "id": "NVDA-prior_year-02"}
{"ticker": "NVDA", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 64089, "prior_answer": 102718, "turns": ["What was NVIDIA Corporation's net cash provided by (used in) operating activities for fiscal 2025, as reported in the comparative figures?"], "id": "NVDA-prior_year-03"}
{"ticker": "NVDA", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 65.47, "turns": ["By what percentage did NVIDIA Corporation's total revenue (net sales) change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "NVDA-yoy-01"}
{"ticker": "NVDA", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 64.75, "turns": ["By what percentage did NVIDIA Corporation's net income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "NVDA-yoy-02"}
{"ticker": "NVDA", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 60.08, "turns": ["By what percentage did NVIDIA Corporation's operating income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "NVDA-yoy-03"}
{"ticker": "NVDA", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 55.6, "turns": ["What was NVIDIA Corporation's net income as a percentage of total revenue for fiscal 2026?"], "id": "NVDA-ratio-01"}
{"ticker": "NVDA", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 60.38, "turns": ["What was NVIDIA Corporation's operating income as a percentage of total revenue for fiscal 2026?"], "id": "NVDA-ratio-02"}
{"ticker": "NVDA", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was NVIDIA Corporation's actual reported total revenue for fiscal year 2029?"], "id": "NVDA-trap-01"}
{"ticker": "NVDA", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did NVIDIA Corporation report for the third quarter of fiscal year 2029?"], "id": "NVDA-trap-02"}
{"ticker": "NVDA", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 130497, "prior_answer": 215938, "turns": ["What was NVIDIA Corporation's total revenue (net sales) for fiscal 2026?", "And what was it in the prior fiscal year?"], "id": "NVDA-multiturn-01"}
{"ticker": "JPM", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 182447, "prior_answer": 177556, "turns": ["What was JPMorgan Chase & Co.'s total revenue (net sales) for 2025?"], "id": "JPM-lookup-01"}
{"ticker": "JPM", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 57048, "prior_answer": 58471, "turns": ["What was JPMorgan Chase & Co.'s net income for 2025?"], "id": "JPM-lookup-02"}
{"ticker": "JPM", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 4424900, "prior_answer": 4002814, "turns": ["What was JPMorgan Chase & Co.'s total assets as of the end of 2025?"], "id": "JPM-lookup-03"}
{"ticker": "JPM", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 362438, "prior_answer": 344758, "turns": ["What was JPMorgan Chase & Co.'s total shareholders' equity as of the end of 2025?"], "id": "JPM-lookup-04"}
{"ticker": "JPM", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": -147782, "prior_answer": -42012, "turns": ["What was JPMorgan Chase & Co.'s net cash provided by (used in) operating activities for 2025?"], "id": "JPM-lookup-05"}
{"ticker": "JPM", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 20.02, "prior_answer": 19.75, "turns": ["What was JPMorgan Chase & Co.'s diluted earnings per share for 2025?"], "id": "JPM-lookup-06"}
{"ticker": "JPM", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 177556, "prior_answer": 182447, "turns": ["What was JPMorgan Chase & Co.'s total revenue (net sales) for 2024, as reported in the comparative figures?"], "id": "JPM-prior_year-01"}
{"ticker": "JPM", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 58471, "prior_answer": 57048, "turns": ["What was JPMorgan Chase & Co.'s net income for 2024, as reported in the comparative figures?"], "id": "JPM-prior_year-02"}
{"ticker": "JPM", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": -42012, "prior_answer": -147782, "turns": ["What was JPMorgan Chase & Co.'s net cash provided by (used in) operating activities for 2024, as reported in the comparative figures?"], "id": "JPM-prior_year-03"}
{"ticker": "JPM", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 2.75, "turns": ["By what percentage did JPMorgan Chase & Co.'s total revenue (net sales) change from 2024 to 2025? Give the percentage change."], "id": "JPM-yoy-01"}
{"ticker": "JPM", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": -2.43, "turns": ["By what percentage did JPMorgan Chase & Co.'s net income change from 2024 to 2025? Give the percentage change."], "id": "JPM-yoy-02"}
{"ticker": "JPM", "type": "yoy", "metric": "op_cash_flow", "unit": "percent", "answer": -251.76, "turns": ["By what percentage did JPMorgan Chase & Co.'s net cash provided by (used in) operating activities change from 2024 to 2025? Give the percentage change."], "id": "JPM-yoy-03"}
{"ticker": "JPM", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 31.27, "turns": ["What was JPMorgan Chase & Co.'s net income as a percentage of total revenue for 2025?"], "id": "JPM-ratio-01"}
{"ticker": "JPM", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was JPMorgan Chase & Co.'s actual reported total revenue for fiscal year 2028?"], "id": "JPM-trap-01"}
{"ticker": "JPM", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did JPMorgan Chase & Co. report for the third quarter of fiscal year 2028?"], "id": "JPM-trap-02"}
{"ticker": "JPM", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 177556, "prior_answer": 182447, "turns": ["What was JPMorgan Chase & Co.'s total revenue (net sales) for 2025?", "And what was it in the prior fiscal year?"], "id": "JPM-multiturn-01"}
{"ticker": "XOM", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 332238, "prior_answer": 349585, "turns": ["What was Exxon Mobil Corporation's total revenue (net sales) for 2025?"], "id": "XOM-lookup-01"}
{"ticker": "XOM", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 28844, "prior_answer": 33680, "turns": ["What was Exxon Mobil Corporation's net income for 2025?"], "id": "XOM-lookup-02"}
{"ticker": "XOM", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 448980, "prior_answer": 453475, "turns": ["What was Exxon Mobil Corporation's total assets as of the end of 2025?"], "id": "XOM-lookup-03"}
{"ticker": "XOM", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 259386, "prior_answer": 263705, "turns": ["What was Exxon Mobil Corporation's total shareholders' equity as of the end of 2025?"], "id": "XOM-lookup-04"}
{"ticker": "XOM", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 51970, "prior_answer": 55022, "turns": ["What was Exxon Mobil Corporation's net cash provided by (used in) operating activities for 2025?"], "id": "XOM-lookup-05"}
{"ticker": "XOM", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 6.7, "prior_answer": 7.84, "turns": ["What was Exxon Mobil Corporation's diluted earnings per share for 2025?"], "id": "XOM-lookup-06"}
{"ticker": "XOM", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 10681, "prior_answer": 23029, "turns": ["What was Exxon Mobil Corporation's cash and cash equivalents as of the end of 2025?"], "id": "XOM-lookup-07"}
{"ticker": "XOM", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 349585, "prior_answer": 332238, "turns": ["What was Exxon Mobil Corporation's total revenue (net sales) for 2024, as reported in the comparative figures?"], "id": "XOM-prior_year-01"}
{"ticker": "XOM", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 33680, "prior_answer": 28844, "turns": ["What was Exxon Mobil Corporation's net income for 2024, as reported in the comparative figures?"], "id": "XOM-prior_year-02"}
{"ticker": "XOM", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 55022, "prior_answer": 51970, "turns": ["What was Exxon Mobil Corporation's net cash provided by (used in) operating activities for 2024, as reported in the comparative figures?"], "id": "XOM-prior_year-03"}
{"ticker": "XOM", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": -4.96, "turns": ["By what percentage did Exxon Mobil Corporation's total revenue (net sales) change from 2024 to 2025? Give the percentage change."], "id": "XOM-yoy-01"}
{"ticker": "XOM", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": -14.36, "turns": ["By what percentage did Exxon Mobil Corporation's net income change from 2024 to 2025? Give the percentage change."], "id": "XOM-yoy-02"}
{"ticker": "XOM", "type": "yoy", "metric": "op_cash_flow", "unit": "percent", "answer": -5.55, "turns": ["By what percentage did Exxon Mobil Corporation's net cash provided by (used in) operating activities change from 2024 to 2025? Give the percentage change."], "id": "XOM-yoy-03"}
{"ticker": "XOM", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 8.68, "turns": ["What was Exxon Mobil Corporation's net income as a percentage of total revenue for 2025?"], "id": "XOM-ratio-01"}
{"ticker": "XOM", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Exxon Mobil Corporation's actual reported total revenue for fiscal year 2028?"], "id": "XOM-trap-01"}
{"ticker": "XOM", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Exxon Mobil Corporation report for the third quarter of fiscal year 2028?"], "id": "XOM-trap-02"}
{"ticker": "XOM", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 349585, "prior_answer": 332238, "turns": ["What was Exxon Mobil Corporation's total revenue (net sales) for 2025?", "And what was it in the prior fiscal year?"], "id": "XOM-multiturn-01"}
{"ticker": "JNJ", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 94193, "prior_answer": 88821, "turns": ["What was Johnson & Johnson's total revenue (net sales) for fiscal 2025?"], "id": "JNJ-lookup-01"}
{"ticker": "JNJ", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 26804, "prior_answer": 14066, "turns": ["What was Johnson & Johnson's net income for fiscal 2025?"], "id": "JNJ-lookup-02"}
{"ticker": "JNJ", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 199210, "prior_answer": 180104, "turns": ["What was Johnson & Johnson's total assets as of the end of fiscal 2025?"], "id": "JNJ-lookup-03"}
{"ticker": "JNJ", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 24530, "prior_answer": 24266, "turns": ["What was Johnson & Johnson's net cash provided by (used in) operating activities for fiscal 2025?"], "id": "JNJ-lookup-04"}
{"ticker": "JNJ", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 11.03, "prior_answer": 5.79, "turns": ["What was Johnson & Johnson's diluted earnings per share for fiscal 2025?"], "id": "JNJ-lookup-05"}
{"ticker": "JNJ", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 19709, "prior_answer": 24105, "turns": ["What was Johnson & Johnson's cash and cash equivalents as of the end of fiscal 2025?"], "id": "JNJ-lookup-06"}
{"ticker": "JNJ", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 88821, "prior_answer": 94193, "turns": ["What was Johnson & Johnson's total revenue (net sales) for fiscal 2024, as reported in the comparative figures?"], "id": "JNJ-prior_year-01"}
{"ticker": "JNJ", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 14066, "prior_answer": 26804, "turns": ["What was Johnson & Johnson's net income for fiscal 2024, as reported in the comparative figures?"], "id": "JNJ-prior_year-02"}
{"ticker": "JNJ", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 24266, "prior_answer": 24530, "turns": ["What was Johnson & Johnson's net cash provided by (used in) operating activities for fiscal 2024, as reported in the comparative figures?"], "id": "JNJ-prior_year-03"}
{"ticker": "JNJ", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 6.05, "turns": ["By what percentage did Johnson & Johnson's total revenue (net sales) change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "JNJ-yoy-01"}
{"ticker": "JNJ", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 90.56, "turns": ["By what percentage did Johnson & Johnson's net income change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "JNJ-yoy-02"}
{"ticker": "JNJ", "type": "yoy", "metric": "op_cash_flow", "unit": "percent", "answer": 1.09, "turns": ["By what percentage did Johnson & Johnson's net cash provided by (used in) operating activities change from fiscal 2024 to fiscal 2025? Give the percentage change."], "id": "JNJ-yoy-03"}
{"ticker": "JNJ", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 28.46, "turns": ["What was Johnson & Johnson's net income as a percentage of total revenue for fiscal 2025?"], "id": "JNJ-ratio-01"}
{"ticker": "JNJ", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Johnson & Johnson's actual reported total revenue for fiscal year 2028?"], "id": "JNJ-trap-01"}
{"ticker": "JNJ", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Johnson & Johnson report for the third quarter of fiscal year 2028?"], "id": "JNJ-trap-02"}
{"ticker": "JNJ", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 88821, "prior_answer": 94193, "turns": ["What was Johnson & Johnson's total revenue (net sales) for fiscal 2025?", "And what was it in the prior fiscal year?"], "id": "JNJ-multiturn-01"}
{"ticker": "WMT", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 706413, "prior_answer": 674538, "turns": ["What was Walmart Inc.'s total revenue (net sales) for fiscal 2026?"], "id": "WMT-lookup-01"}
{"ticker": "WMT", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 21893, "prior_answer": 19436, "turns": ["What was Walmart Inc.'s net income for fiscal 2026?"], "id": "WMT-lookup-02"}
{"ticker": "WMT", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 284668, "prior_answer": 260823, "turns": ["What was Walmart Inc.'s total assets as of the end of fiscal 2026?"], "id": "WMT-lookup-03"}
{"ticker": "WMT", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 99617, "prior_answer": 91013, "turns": ["What was Walmart Inc.'s total shareholders' equity as of the end of fiscal 2026?"], "id": "WMT-lookup-04"}
{"ticker": "WMT", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 41565, "prior_answer": 36443, "turns": ["What was Walmart Inc.'s net cash provided by (used in) operating activities for fiscal 2026?"], "id": "WMT-lookup-05"}
{"ticker": "WMT", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 10727, "prior_answer": 9037, "turns": ["What was Walmart Inc.'s cash and cash equivalents as of the end of fiscal 2026?"], "id": "WMT-lookup-06"}
{"ticker": "WMT", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 29825, "prior_answer": 29348, "turns": ["What was Walmart Inc.'s operating income for fiscal 2026?"], "id": "WMT-lookup-07"}
{"ticker": "WMT", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 674538, "prior_answer": 706413, "turns": ["What was Walmart Inc.'s total revenue (net sales) for fiscal 2025, as reported in the comparative figures?"], "id": "WMT-prior_year-01"}
{"ticker": "WMT", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 19436, "prior_answer": 21893, "turns": ["What was Walmart Inc.'s net income for fiscal 2025, as reported in the comparative figures?"], "id": "WMT-prior_year-02"}
{"ticker": "WMT", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 36443, "prior_answer": 41565, "turns": ["What was Walmart Inc.'s net cash provided by (used in) operating activities for fiscal 2025, as reported in the comparative figures?"], "id": "WMT-prior_year-03"}
{"ticker": "WMT", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 4.73, "turns": ["By what percentage did Walmart Inc.'s total revenue (net sales) change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "WMT-yoy-01"}
{"ticker": "WMT", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 12.64, "turns": ["By what percentage did Walmart Inc.'s net income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "WMT-yoy-02"}
{"ticker": "WMT", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 1.63, "turns": ["By what percentage did Walmart Inc.'s operating income change from fiscal 2025 to fiscal 2026? Give the percentage change."], "id": "WMT-yoy-03"}
{"ticker": "WMT", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 3.1, "turns": ["What was Walmart Inc.'s net income as a percentage of total revenue for fiscal 2026?"], "id": "WMT-ratio-01"}
{"ticker": "WMT", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 4.22, "turns": ["What was Walmart Inc.'s operating income as a percentage of total revenue for fiscal 2026?"], "id": "WMT-ratio-02"}
{"ticker": "WMT", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Walmart Inc.'s actual reported total revenue for fiscal year 2029?"], "id": "WMT-trap-01"}
{"ticker": "WMT", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Walmart Inc. report for the third quarter of fiscal year 2029?"], "id": "WMT-trap-02"}
{"ticker": "WMT", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 674538, "prior_answer": 706413, "turns": ["What was Walmart Inc.'s total revenue (net sales) for fiscal 2026?", "And what was it in the prior fiscal year?"], "id": "WMT-multiturn-01"}
{"ticker": "CAT", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 67589, "prior_answer": 64809, "turns": ["What was Caterpillar Inc.'s total revenue (net sales) for 2025?"], "id": "CAT-lookup-01"}
{"ticker": "CAT", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 98585, "prior_answer": 87764, "turns": ["What was Caterpillar Inc.'s total assets as of the end of 2025?"], "id": "CAT-lookup-02"}
{"ticker": "CAT", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 11739, "prior_answer": 12035, "turns": ["What was Caterpillar Inc.'s net cash provided by (used in) operating activities for 2025?"], "id": "CAT-lookup-03"}
{"ticker": "CAT", "type": "lookup", "metric": "rnd", "unit": "usd_millions", "answer": 2148, "prior_answer": 2107, "turns": ["What was Caterpillar Inc.'s research and development expense for 2025?"], "id": "CAT-lookup-04"}
{"ticker": "CAT", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 18.81, "prior_answer": 22.05, "turns": ["What was Caterpillar Inc.'s diluted earnings per share for 2025?"], "id": "CAT-lookup-05"}
{"ticker": "CAT", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 9980, "prior_answer": 6889, "turns": ["What was Caterpillar Inc.'s cash and cash equivalents as of the end of 2025?"], "id": "CAT-lookup-06"}
{"ticker": "CAT", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 11151, "prior_answer": 13072, "turns": ["What was Caterpillar Inc.'s operating income for 2025?"], "id": "CAT-lookup-07"}
{"ticker": "CAT", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 64809, "prior_answer": 67589, "turns": ["What was Caterpillar Inc.'s total revenue (net sales) for 2024, as reported in the comparative figures?"], "id": "CAT-prior_year-01"}
{"ticker": "CAT", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 12035, "prior_answer": 11739, "turns": ["What was Caterpillar Inc.'s net cash provided by (used in) operating activities for 2024, as reported in the comparative figures?"], "id": "CAT-prior_year-02"}
{"ticker": "CAT", "type": "prior_year", "metric": "total_assets", "unit": "usd_millions", "answer": 87764, "prior_answer": 98585, "turns": ["What was Caterpillar Inc.'s total assets as of the end of 2024, as reported in the comparative figures?"], "id": "CAT-prior_year-03"}
{"ticker": "CAT", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 4.29, "turns": ["By what percentage did Caterpillar Inc.'s total revenue (net sales) change from 2024 to 2025? Give the percentage change."], "id": "CAT-yoy-01"}
{"ticker": "CAT", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": -14.7, "turns": ["By what percentage did Caterpillar Inc.'s operating income change from 2024 to 2025? Give the percentage change."], "id": "CAT-yoy-02"}
{"ticker": "CAT", "type": "yoy", "metric": "op_cash_flow", "unit": "percent", "answer": -2.46, "turns": ["By what percentage did Caterpillar Inc.'s net cash provided by (used in) operating activities change from 2024 to 2025? Give the percentage change."], "id": "CAT-yoy-03"}
{"ticker": "CAT", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 16.5, "turns": ["What was Caterpillar Inc.'s operating income as a percentage of total revenue for 2025?"], "id": "CAT-ratio-01"}
{"ticker": "CAT", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Caterpillar Inc.'s actual reported total revenue for fiscal year 2028?"], "id": "CAT-trap-01"}
{"ticker": "CAT", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Caterpillar Inc. report for the third quarter of fiscal year 2028?"], "id": "CAT-trap-02"}
{"ticker": "CAT", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 64809, "prior_answer": 67589, "turns": ["What was Caterpillar Inc.'s total revenue (net sales) for 2025?", "And what was it in the prior fiscal year?"], "id": "CAT-multiturn-01"}
{"ticker": "KO", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 47941, "prior_answer": 47061, "turns": ["What was The Coca-Cola Company's total revenue (net sales) for 2025?"], "id": "KO-lookup-01"}
{"ticker": "KO", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 13107, "prior_answer": 10631, "turns": ["What was The Coca-Cola Company's net income for 2025?"], "id": "KO-lookup-02"}
{"ticker": "KO", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 104816, "prior_answer": 100549, "turns": ["What was The Coca-Cola Company's total assets as of the end of 2025?"], "id": "KO-lookup-03"}
{"ticker": "KO", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 32169, "prior_answer": 24856, "turns": ["What was The Coca-Cola Company's total shareholders' equity as of the end of 2025?"], "id": "KO-lookup-04"}
{"ticker": "KO", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 7408, "prior_answer": 6805, "turns": ["What was The Coca-Cola Company's net cash provided by (used in) operating activities for 2025?"], "id": "KO-lookup-05"}
{"ticker": "KO", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 10270, "prior_answer": 10828, "turns": ["What was The Coca-Cola Company's cash and cash equivalents as of the end of 2025?"], "id": "KO-lookup-06"}
{"ticker": "KO", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 13762, "prior_answer": 9992, "turns": ["What was The Coca-Cola Company's operating income for 2025?"], "id": "KO-lookup-07"}
{"ticker": "KO", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 47061, "prior_answer": 47941, "turns": ["What was The Coca-Cola Company's total revenue (net sales) for 2024, as reported in the comparative figures?"], "id": "KO-prior_year-01"}
{"ticker": "KO", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 10631, "prior_answer": 13107, "turns": ["What was The Coca-Cola Company's net income for 2024, as reported in the comparative figures?"], "id": "KO-prior_year-02"}
{"ticker": "KO", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 6805, "prior_answer": 7408, "turns": ["What was The Coca-Cola Company's net cash provided by (used in) operating activities for 2024, as reported in the comparative figures?"], "id": "KO-prior_year-03"}
{"ticker": "KO", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 1.87, "turns": ["By what percentage did The Coca-Cola Company's total revenue (net sales) change from 2024 to 2025? Give the percentage change."], "id": "KO-yoy-01"}
{"ticker": "KO", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 23.29, "turns": ["By what percentage did The Coca-Cola Company's net income change from 2024 to 2025? Give the percentage change."], "id": "KO-yoy-02"}
{"ticker": "KO", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 37.73, "turns": ["By what percentage did The Coca-Cola Company's operating income change from 2024 to 2025? Give the percentage change."], "id": "KO-yoy-03"}
{"ticker": "KO", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 27.34, "turns": ["What was The Coca-Cola Company's net income as a percentage of total revenue for 2025?"], "id": "KO-ratio-01"}
{"ticker": "KO", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 28.71, "turns": ["What was The Coca-Cola Company's operating income as a percentage of total revenue for 2025?"], "id": "KO-ratio-02"}
{"ticker": "KO", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was The Coca-Cola Company's actual reported total revenue for fiscal year 2028?"], "id": "KO-trap-01"}
{"ticker": "KO", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did The Coca-Cola Company report for the third quarter of fiscal year 2028?"], "id": "KO-trap-02"}
{"ticker": "KO", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 47061, "prior_answer": 47941, "turns": ["What was The Coca-Cola Company's total revenue (net sales) for 2025?", "And what was it in the prior fiscal year?"], "id": "KO-multiturn-01"}
{"ticker": "NFLX", "type": "lookup", "metric": "revenue", "unit": "usd_millions", "answer": 45183.036, "prior_answer": 39000.966, "turns": ["What was Netflix, Inc.'s total revenue (net sales) for 2025?"], "id": "NFLX-lookup-01"}
{"ticker": "NFLX", "type": "lookup", "metric": "net_income", "unit": "usd_millions", "answer": 10981.201, "prior_answer": 8711.631, "turns": ["What was Netflix, Inc.'s net income for 2025?"], "id": "NFLX-lookup-02"}
{"ticker": "NFLX", "type": "lookup", "metric": "total_assets", "unit": "usd_millions", "answer": 55596.993, "prior_answer": 53630.374, "turns": ["What was Netflix, Inc.'s total assets as of the end of 2025?"], "id": "NFLX-lookup-03"}
{"ticker": "NFLX", "type": "lookup", "metric": "equity", "unit": "usd_millions", "answer": 26615.488, "prior_answer": 24743.567, "turns": ["What was Netflix, Inc.'s total shareholders' equity as of the end of 2025?"], "id": "NFLX-lookup-04"}
{"ticker": "NFLX", "type": "lookup", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 10149.273, "prior_answer": 7361.364, "turns": ["What was Netflix, Inc.'s net cash provided by (used in) operating activities for 2025?"], "id": "NFLX-lookup-05"}
{"ticker": "NFLX", "type": "lookup", "metric": "rnd", "unit": "usd_millions", "answer": 3391.39, "prior_answer": 2925.295, "turns": ["What was Netflix, Inc.'s research and development expense for 2025?"], "id": "NFLX-lookup-06"}
{"ticker": "NFLX", "type": "lookup", "metric": "eps_diluted", "unit": "usd_per_share", "answer": 2.53, "prior_answer": 1.98, "turns": ["What was Netflix, Inc.'s diluted earnings per share for 2025?"], "id": "NFLX-lookup-07"}
{"ticker": "NFLX", "type": "lookup", "metric": "cash", "unit": "usd_millions", "answer": 9033.681, "prior_answer": 7804.733, "turns": ["What was Netflix, Inc.'s cash and cash equivalents as of the end of 2025?"], "id": "NFLX-lookup-08"}
{"ticker": "NFLX", "type": "lookup", "metric": "operating_income", "unit": "usd_millions", "answer": 13326.603, "prior_answer": 10417.614, "turns": ["What was Netflix, Inc.'s operating income for 2025?"], "id": "NFLX-lookup-09"}
{"ticker": "NFLX", "type": "prior_year", "metric": "revenue", "unit": "usd_millions", "answer": 39000.966, "prior_answer": 45183.036, "turns": ["What was Netflix, Inc.'s total revenue (net sales) for 2024, as reported in the comparative figures?"], "id": "NFLX-prior_year-01"}
{"ticker": "NFLX", "type": "prior_year", "metric": "net_income", "unit": "usd_millions", "answer": 8711.631, "prior_answer": 10981.201, "turns": ["What was Netflix, Inc.'s net income for 2024, as reported in the comparative figures?"], "id": "NFLX-prior_year-02"}
{"ticker": "NFLX", "type": "prior_year", "metric": "op_cash_flow", "unit": "usd_millions", "answer": 7361.364, "prior_answer": 10149.273, "turns": ["What was Netflix, Inc.'s net cash provided by (used in) operating activities for 2024, as reported in the comparative figures?"], "id": "NFLX-prior_year-03"}
{"ticker": "NFLX", "type": "yoy", "metric": "revenue", "unit": "percent", "answer": 15.85, "turns": ["By what percentage did Netflix, Inc.'s total revenue (net sales) change from 2024 to 2025? Give the percentage change."], "id": "NFLX-yoy-01"}
{"ticker": "NFLX", "type": "yoy", "metric": "net_income", "unit": "percent", "answer": 26.05, "turns": ["By what percentage did Netflix, Inc.'s net income change from 2024 to 2025? Give the percentage change."], "id": "NFLX-yoy-02"}
{"ticker": "NFLX", "type": "yoy", "metric": "operating_income", "unit": "percent", "answer": 27.92, "turns": ["By what percentage did Netflix, Inc.'s operating income change from 2024 to 2025? Give the percentage change."], "id": "NFLX-yoy-03"}
{"ticker": "NFLX", "type": "ratio", "metric": "net_income_margin", "unit": "percent", "answer": 24.3, "turns": ["What was Netflix, Inc.'s net income as a percentage of total revenue for 2025?"], "id": "NFLX-ratio-01"}
{"ticker": "NFLX", "type": "ratio", "metric": "operating_income_margin", "unit": "percent", "answer": 29.49, "turns": ["What was Netflix, Inc.'s operating income as a percentage of total revenue for 2025?"], "id": "NFLX-ratio-02"}
{"ticker": "NFLX", "type": "trap", "metric": "future_revenue", "unit": "abstain", "answer": null, "turns": ["What was Netflix, Inc.'s actual reported total revenue for fiscal year 2028?"], "id": "NFLX-trap-01"}
{"ticker": "NFLX", "type": "trap", "metric": "future_quarter", "unit": "abstain", "answer": null, "turns": ["What diluted EPS did Netflix, Inc. report for the third quarter of fiscal year 2028?"], "id": "NFLX-trap-02"}
{"ticker": "NFLX", "type": "multiturn", "metric": "revenue", "unit": "usd_millions", "answer": 39000.966, "prior_answer": 45183.036, "turns": ["What was Netflix, Inc.'s total revenue (net sales) for 2025?", "And what was it in the prior fiscal year?"], "id": "NFLX-multiturn-01"}


In [ ]:
%%writefile models.yaml
# Any OpenAI-compatible endpoint. Keys come from the env var named in api_key_env.
# Pick 2 frontier + 3 open models. Model IDs change often — check the provider's model list first.
models:
  - name: gpt-4.1-mini
    model: gpt-4.1-mini
    api_key_env: OPENAI_API_KEY

  - name: claude-via-openrouter
    model: anthropic/claude-sonnet-4.5     # replace with a current OpenRouter model id
    base_url: https://openrouter.ai/api/v1
    api_key_env: OPENROUTER_API_KEY

  - name: llama-3.1-8b
    model: meta-llama/llama-3.1-8b-instruct
    base_url: https://openrouter.ai/api/v1
    api_key_env: OPENROUTER_API_KEY

  - name: qwen-2.5-7b
    model: qwen/qwen-2.5-7b-instruct
    base_url: https://openrouter.ai/api/v1
    api_key_env: OPENROUTER_API_KEY

  # Local vLLM (Kaggle T4 / your GPU):  vllm serve Qwen/Qwen2.5-7B-Instruct --max-model-len 16384
  # - name: qwen-local
  #   model: Qwen/Qwen2.5-7B-Instruct
  #   base_url: http://localhost:8000/v1
  #   api_key_env: VLLM_KEY


In [ ]:
import os
os.environ['SEC_USER_AGENT'] = 'Vedansh Mathur vedanshmathur007@gmail.com'
!python -m finhallu.corpus

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m finhallu.run --hf Qwen/Qwen2.5-7B-Instruct microsoft/Phi-3.5-mini-instruct ibm-granite/granite-3.1-8b-instruct

In [ ]:
!python -m finhallu.report results/results.csv
!head -c 3000 results/results.csv